# Post-APA calling: Imputation and QC
## Description
This notebook is designed to impute the missing values in the PDUI matrix, and perform quantile normailization for the impute values.
## Input
* raw PDUI matrix (row as gene, columns as sample id)
* covariate file  

## Output
* PDUI matrix without missingness  
  - The missing value is calculated using `impute` package

## Steps

**Timing:** Runtime varies by dataset size and compute resources. For the toy chr22 MWE dataset, most steps complete in under 10 minutes on a standard HPC node.

In [ ]:
sos run pipeline/apa_impute.ipynb APAimpute \
    --cwd output/apa \
    --chrlist chr22

Optionally, rename the sample columns of the imputed PDUI matrix using a match table (maps the Dapars sample IDs to your desired sample names):

In [ ]:
sos run pipeline/apa_impute.ipynb APArename \
    --cwd output/apa \
    --chrlist chr22 \
    --match input/covariate/protocol_example.apa_matchtable.txt

## Command interface

In [ ]:
sos run pipeline/apa_impute.ipynb -h

## Workflow implementation

## Anticipated Results

The pipeline produces output files in the `output/` subdirectory named after the workflow step. Verify success by checking that output files exist and are non-empty. See the **Output** section above for the expected file names and formats.

In [ ]:
[global]
parameter: walltime = '40h'
parameter: mem = '32G'
parameter: ncore = 16
parameter: cwd = path
parameter: modular_script_dir = path('code/script')  # override with --modular-script-dir
parameter: thread = 8
parameter: job_size = 1
parameter: container = ''

In [ ]:
[APAimpute]
parameter: chrlist = list
input: [f'{cwd}/apa_{x}/Dapars_result_result_temp.{x}.txt' for x in chrlist]
output: [f'{cwd}/apa_{x}/Dapars_result_impute_{x}.bed' for x in chrlist]
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = ncore
bash: expand= "${ }"
    Rscript ${modular_script_dir}/molecular_phenotypes/QC/apa_impute.R --step impute \
        --cwd ${cwd} \
        --chrlist "${chrlist}"

In [ ]:
[APArename_1]
parameter: match = path
parameter: chrlist = list
input: [f'{cwd}/apa_{x}/Dapars_result_impute_{x}.bed' for x in chrlist], group_by = 1
output: [f'{cwd}/apa_{x}/Dapars_result_impute_renamed_{x}.bed' for x in chrlist], group_by = 1
bash: expand= "${ }"
    Rscript ${modular_script_dir}/molecular_phenotypes/QC/apa_impute.R --step rename \
        --input "${_input}" \
        --match "${match}" \
        --output "${_output}" \
        --update-end

In [ ]:
[APArename_2]
parameter: match = path
parameter: chrlist = list
input: [f'{cwd}/apa_{x}/Dapars_result_impute_renamed_{x}.bed' for x in chrlist], group_by = 1
output: [f'{cwd}/apa_{x}/Dapars_result_impute_renamed_{x}.bed.gz' for x in chrlist], group_by = 1
bash: expand= "${ }"
    Rscript ${modular_script_dir}/molecular_phenotypes/QC/apa_impute.R --step bgzip_index \
        --input "${_input}" \
        --output "${_output}"

In [ ]:
[APArename_3]
parameter: match = "input/covariate/protocol_example.apa_matchtable.txt" #path
input: f'{cwd}/Dapars_allchrom.bed'
output: f'{cwd}/Dapars_allchrom_renamed.bed'
bash: expand= "${ }"
    Rscript ${modular_script_dir}/molecular_phenotypes/QC/apa_impute.R --step rename \
        --input "${_input}" \
        --match "${match}" \
        --output "${_output}"

[APArename_4]
output: f'{_input}.gz', f'{_input}.gz.tbi'
bash: expand = "${ }"
    Rscript ${modular_script_dir}/molecular_phenotypes/QC/apa_impute.R --step bgzip_index \
        --input "${_input}" \
        --output "${_output[0]}"